# Exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import random
import time
import tracemalloc
from scipy.special import factorial
from scipy import stats
import heapq
from scipy.stats import chisquare
import pandas as pd
import math
from scipy.stats import norm

## Exercise 1

## 1. LCG

In [ ]:
def LCG(x0, a, c, M, N):
    xmi = x0
    rando, rando_Ui = [], []
    rando.append(xmi)
    rando_Ui.append(xmi/M)
    for i in range(N):
        xi = (a*xmi+c) % M
        rando.append(xi)
        rando_Ui.append(xi/M)
        xmi = xi
    return rando, rando_Ui

### (a) - Generate 10.000 random numbers

In [ ]:
import matplotlib.pyplot as plt
a = 5
c = 1
M = 16
x0 = 3
N = 10000
rando, rando_Ui = LCG(x0, a, c, M, N-1)
plt.hist(rando, bins=10)

### (b) Evaluate

In [ ]:
# Scatterplot Ui vs Ui+1
ri = rando[:-1]
ri_plus_1 = rando[1:]
plt.scatter(ri, ri_plus_1)
plt.xlabel('Ui')
plt.ylabel('Ui+1')
plt.title('Scatterplot of Ui vs Ui+1')
plt.show()

In [ ]:
# Scatterplot Ui vs Ui+1
Ui = rando_Ui[:-1]
Ui_plus_1 = rando_Ui[1:]
plt.scatter(Ui, Ui_plus_1)
plt.xlabel('Ui')
plt.ylabel('Ui+1')
plt.title('Scatterplot of Ui vs Ui+1')
plt.show()

In [ ]:
# Chi-squared test
def chi_squared_test(observed, expected):
    T = sum((o - e) ** 2 / e for o, e in zip(observed, expected))
    df = len(observed) - 1
    return T, df

In [ ]:
import numpy as np
from scipy.stats import chi2

observed, _ = np.histogram(rando_Ui, bins=10)
expected = [len(rando_Ui) / 10] * 10
T,df = chi_squared_test(observed, expected)
print(f"Chi-squared test statistic: {T}")
print(f"Degrees of freedom: {df}")

critical_value = chi2.ppf(0.95, df)
print(f"Critical value at 95% confidence: {critical_value}")

if T > critical_value:
    print("Forkast H0")
else:
    print("Kan ikke forkaste H0")

In [ ]:
# Kolmogorov-Smirnov test givet uniform
def KS_test(data, case):
    data = np.sort(data)
    n = len(data)

    Dn = 0

    for i, x in enumerate(data, start=1):
        Fn = i / n
        F = x  # For uniform distribution
        Dn = max(Dn, abs(Fn - F))
    
    if case == 'known':
        adjusted_test_stat = (np.sqrt(n) + 0.12 + 0.11/np.sqrt(n))*Dn
    elif case == 'normal':
        adjusted_test_stat = (np.sqrt(n) - 0.01 + 0.85/np.sqrt(n))*Dn
    elif case == 'exp':
        adjusted_test_stat = (np.sqrt(n)+0.26+0.5/np.sqrt(n))*(Dn-0.2/n)

    print(f"Kolmogorov-Smirnov test statistic: {Dn}")
    print(f"Adjusted Kolmogorov-Smirnov test statistic: {adjusted_test_stat}")
    
    return Dn, adjusted_test_stat

In [ ]:
KS_test(rando_Ui, 'known')

Da den er højere end 1.628, betyder det at selv for significance level 0.01, kan vi afvise null-hypotesen om at tallene er uniformt fordelt.

In [ ]:
# run-tests

def run_wald_wolf(data):
    median = np.median(data)

    # Indel data i to sekvenser baseret på medianen
    seq = ['A' if x > median else 'B' for x in data]

    count_below = seq.count('A')
    count_above = seq.count('B')

    # Tæl runs
    T = 1
    for i in range(1, len(seq)):
        if seq[i] != seq[i-1]:
            T += 1
    
    # Udregn forventning og varians
    expectation = 2*(count_above * count_below) / (count_above + count_below)
    variance = (2*count_above * count_below * (2*count_above*count_below - count_above - count_below)) / ((count_above + count_below)**2 * (count_above + count_below - 1))

    z = (T - expectation) / np.sqrt(variance)
    print(f"Number of runs: {T}")
    print(f"Z-score: {z}")

    return T,(T-expectation) / np.sqrt(variance)


In [ ]:
run_wald_wolf(rando_Ui)

In [ ]:
def run_knuth(data):
    T= 1
    run_length = []
    start = 1
    for i in range(1,len(data)):
        if data[i] < data[i-1]:
            T += 1
            run_length.append(start)
            start = 1
        else:
            start += 1
    run_length.append(start)

    R = np.zeros(6)

    for r in run_length:
        if r >= 6:
            R[5] += 1
        else:
            R[r - 1] += 1

    n = len(data)

    B = np.array([
        1/6,
        5/24,
        11/120,
        19/720,
        29/5040,
        1/840
    ])

    A = np.array([
        [4529.4, 9044.9, 13568, 18091, 22615, 27892],
        [9044.9, 18097, 27139, 36187, 45234, 55789],
        [13568, 27139, 40721, 54281, 67852, 83685],
        [18091, 36187, 54281, 72414, 90470, 111580],
        [22615, 45234, 67852, 90470, 113262, 139476],
        [27892, 55789, 83685, 111580, 139476, 172860]
    ])

    D = R - n * B

    Z = (1 / (n - 6)) * (D.T @ A @ D)

    print(f"Number of runs: {T}")
    print(f"Z-score: {Z}")

    return T, Z

In [ ]:
run_knuth(rando_Ui)

In [ ]:
chi2.ppf(0.99, 6)
print(f"Critical value at 99% confidence: {chi2.ppf(0.99, 6)}")

In [ ]:
def run_up_down(data):
    signs = []

    for i in range(1, len(data)):
        if data[i] < data[i-1]:
            signs.append('<')
        else:
            signs.append('>')

    run_length = []
    current = 1
    for i in range(1, len(signs)):
        if signs[i] == signs[i-1]:
            current += 1
        else:
            run_length.append(current)
            current = 1

    run_length.append(current)

    runs = len(run_length)
    n = len(data)
    expected_runs = (2*n - 1) / 3
    variance_runs = (16*n - 29) / 90

    Z = (runs - expected_runs) / (np.sqrt(variance_runs))

    print(f"Number of runs: {runs}")
    print(f"Z-score: {Z}")

    return runs, Z

In [ ]:
run_up_down(rando_Ui)

In [ ]:
def correlation_test(data, lag):
    n = len(data)
    h = lag

    sum = 0
    for i in range(1, n-h):
        sum += data[i]*data[i+h]

    Ch_calculated = 1/(n-h) * sum
    
    expectation = 1/4
    variance = 7/(144*n)

    z = (Ch_calculated-expectation)/np.sqrt(variance)

    return Ch_calculated, z


In [ ]:
for h in [1,2,5,10]:
    print(correlation_test(rando_Ui, h))

## 2. SAG

In [ ]:
np.random.seed(42)
data = np.random.rand(10000)

In [ ]:
plt.hist(rando, bins=10)

In [ ]:
# Scatterplot Ui vs Ui+1
Ui = data[:-1]
Ui_plus_1 = data[1:]
plt.scatter(Ui, Ui_plus_1)
plt.xlabel('Ui')
plt.ylabel('Ui+1')
plt.title('Scatterplot of Ui vs Ui+1')
plt.show()

In [ ]:
import numpy as np
from scipy.stats import chi2

observed, _ = np.histogram(data, bins=10)
expected = [len(data) / 10] * 10
T,df = chi_squared_test(observed, expected)
print(f"Chi-squared test statistic: {T}")
print(f"Degrees of freedom: {df}")

critical_value = chi2.ppf(0.95, df)
print(f"Critical value at 95% confidence: {critical_value}")

if T > critical_value:
    print("Forkast H0")
else:
    print("Kan ikke forkaste H0")

In [ ]:
KS_test(data, 'known')

In [ ]:
run_wald_wolf(data)

In [ ]:
run_knuth(data)

In [ ]:
chi2.ppf(0.95, 6)

In [ ]:
run_up_down(data)

In [ ]:
for h in [1,2,5,10]:
    print(correlation_test(data, h))

## 3. Discuss

In [ ]:
class RandomnessTests:

    @staticmethod
    def LCG(x0, a, c, M, N):
        xmi = x0
        rando, rando_Ui = [], []

        rando.append(xmi)
        rando_Ui.append(xmi / M)

        for _ in range(N):
            xi = (a * xmi + c) % M
            rando.append(xi)
            rando_Ui.append(xi / M)
            xmi = xi

        return rando, rando_Ui

    @staticmethod
    def chi_squared_test(observed, expected):
        T = sum((o - e) ** 2 / e for o, e in zip(observed, expected))
        df = len(observed) - 1
        return T, df

    @staticmethod
    def KS_test(data, case):
        data = np.sort(data)
        n = len(data)

        Dn = 0

        for i, x in enumerate(data, start=1):
            Fn = i / n
            F = x  # Uniform distribution
            Dn = max(Dn, abs(Fn - F))

        if case == 'known':
            adjusted_test_stat = (
                np.sqrt(n) + 0.12 + 0.11 / np.sqrt(n)
            ) * Dn
        elif case == 'normal':
            adjusted_test_stat = (
                np.sqrt(n) - 0.01 + 0.85 / np.sqrt(n)
            ) * Dn
        elif case == 'exp':
            adjusted_test_stat = (
                np.sqrt(n) + 0.26 + 0.5 / np.sqrt(n)
            ) * (Dn - 0.2 / n)
        else:
            raise ValueError(
                "case skal være 'known', 'normal' eller 'exp'"
            )

        print(f"Kolmogorov-Smirnov test statistic: {Dn}")
        print(
            f"Adjusted Kolmogorov-Smirnov test statistic: "
            f"{adjusted_test_stat}"
        )

        return Dn, adjusted_test_stat

    @staticmethod
    def run_wald_wolf(data):
        median = np.median(data)

        seq = ['A' if x > median else 'B' for x in data]

        count_below = seq.count('A')
        count_above = seq.count('B')

        T = 1
        for i in range(1, len(seq)):
            if seq[i] != seq[i - 1]:
                T += 1

        expectation = (
            2 * (count_above * count_below)
            / (count_above + count_below)
        )

        variance = (
            2 * count_above * count_below *
            (2 * count_above * count_below -
             count_above - count_below)
        ) / (
            (count_above + count_below) ** 2 *
            (count_above + count_below - 1)
        )

        z = (T - expectation) / np.sqrt(variance)

        print(f"Number of runs: {T}")
        print(f"Z-score: {z}")

        return T, z

    @staticmethod
    def run_knuth(data):
        T = 1
        run_length = []
        start = 1

        for i in range(1, len(data)):
            if data[i] < data[i - 1]:
                T += 1
                run_length.append(start)
                start = 1
            else:
                start += 1

        run_length.append(start)

        R = np.zeros(6)

        for r in run_length:
            if r >= 6:
                R[5] += 1
            else:
                R[r - 1] += 1

        n = len(data)

        B = np.array([
            1 / 6,
            5 / 24,
            11 / 120,
            19 / 720,
            29 / 5040,
            1 / 840
        ])

        A = np.array([
            [4529.4, 9044.9, 13568, 18091, 22615, 27892],
            [9044.9, 18097, 27139, 36187, 45234, 55789],
            [13568, 27139, 40721, 54281, 67852, 83685],
            [18091, 36187, 54281, 72414, 90470, 111580],
            [22615, 45234, 67852, 90470, 113262, 139476],
            [27892, 55789, 83685, 111580, 139476, 172860]
        ])

        D = R - n * B

        Z = (1 / (n - 6)) * (D.T @ A @ D)

        print(f"Number of runs: {T}")
        print(f"Z-score: {Z}")

        return T, Z

    @staticmethod
    def run_up_down(data):
        signs = []

        for i in range(1, len(data)):
            if data[i] < data[i - 1]:
                signs.append('<')
            else:
                signs.append('>')

        run_length = []
        current = 1

        for i in range(1, len(signs)):
            if signs[i] == signs[i - 1]:
                current += 1
            else:
                run_length.append(current)
                current = 1

        run_length.append(current)

        runs = len(run_length)
        n = len(data)

        expected_runs = (2 * n - 1) / 3
        variance_runs = (16 * n - 29) / 90

        Z = (
            runs - expected_runs
        ) / np.sqrt(variance_runs)

        print(f"Number of runs: {runs}")
        print(f"Z-score: {Z}")

        return runs, Z

    @staticmethod
    def correlation_test(data, lag):
        n = len(data)

        total = 0
        for i in range(1, n - lag):
            total += data[i] * data[i + lag]

        Ch_calculated = total / (n - lag)

        expectation = 1 / 4
        variance = 7 / (144 * n)

        z = (
            Ch_calculated - expectation
        ) / np.sqrt(variance)

        return Ch_calculated, z
    
    @staticmethod
    def analyze(
        data,
        bins=10,
        ks_case="known",
        correlation_lags=(1, 2, 5, 10),
        histogram=True,
        scatter=True,
        chi_square=True,
        ks=True,
        wald_wolf=True,
        knuth=True,
        up_down=True,
        correlation=True,
    ):
        if histogram:
            plt.figure()
            plt.hist(data, bins=bins)
            plt.title("Histogram")
            plt.show()

        if scatter:
            plt.figure()
            plt.scatter(data[:-1], data[1:])
            plt.xlabel("Ui")
            plt.ylabel("Ui+1")
            plt.title("Scatterplot")
            plt.show()

        results = {}

        if chi_square:
            observed, _ = np.histogram(data, bins=bins)
            expected = [len(data) / bins] * bins

            T, df = RandomnessTests.chi_squared_test(observed, expected)
            results["chi_square"] = (T, df)

        if ks:
            results["ks"] = RandomnessTests.KS_test(data, ks_case)

        if wald_wolf:
            results["wald_wolf"] = RandomnessTests.run_wald_wolf(data)

        if knuth:
            results["knuth"] = RandomnessTests.run_knuth(data)

        if up_down:
            results["up_down"] = RandomnessTests.run_up_down(data)

        if correlation:
            corr_results = []
            for lag in correlation_lags:
                corr_results.append(
                    RandomnessTests.correlation_test(data, lag)
                )
            results["correlation"] = corr_results

        return results
                
    @staticmethod
    def compare_datasets(datasets, labels=None, **kwargs):
        if labels is None:
            labels = [f"data_{i}" for i in range(len(datasets))]

        all_results = {}

        for label, data in zip(labels, datasets):
            print(f"\n {label} ")
            all_results[label] = RandomnessTests.analyze(data, **kwargs)

        return all_results

In [ ]:
p = 0.1
X = np.random.geometric(p, size=10000)
plt.hist(X, bins=10)
# Scatterplot Ui vs Ui+1
Ui = X[:-1]
Ui_plus_1 = X[1:]
plt.scatter(Ui, Ui_plus_1)
plt.xlabel('Ui')
plt.ylabel('Ui+1')
plt.title('Scatterplot of Ui vs Ui+1')
plt.show()
observed, _ = np.histogram(X, bins=10)
expected = [len(X) / 10] * 10
RandomnessTests.chi_squared_test(observed, expected)
RandomnessTests.KS_test(X, 'known')
RandomnessTests.run_wald_wolf(X)
RandomnessTests.run_knuth(X)
RandomnessTests.run_up_down(X)
for h in [1,2,5,10]:
    print(RandomnessTests.correlation_test(X, h))

## Exercise 2

In [ ]:
np.random.seed(42)

def sample_geometric(p, n):
    np.random.seed(42)
    U = np.random.uniform(0, 1, n)
    return np.floor(np.log(U) / np.log(1 - p)).astype(int) + 1

p01 = sample_geometric(0.1, 10000)
p05 = sample_geometric(0.5, 10000)
p09 = sample_geometric(0.9, 10000)

In [ ]:
def plot_geometric(samples, p, ax):
    k_max = samples.max() + 1
    k = np.arange(1, k_max + 1)
    pmf = (1 - p) ** (k - 1) * p  # geometric PMF

    # Normalize histogram to probability mass (density=False, divide by n)
    counts, _ = np.histogram(samples, bins=np.arange(1, k_max + 2) - 0.5)
    ax.bar(k, counts / len(samples), width=0.6, alpha=0.6, label="Simulated")
    ax.plot(k, pmf, "ro-", markersize=4, linewidth=1.5, label="Theoretical PMF")
    ax.set_title(f"Geometric distribution  (p = {p})")
    ax.set_xlabel("k")
    ax.set_ylabel("Probability")
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (samples, p) in zip(axes, [(p01, 0.1), (p05, 0.5), (p09, 0.9)]):
    plot_geometric(samples, p, ax)

plt.tight_layout()
plt.show()


In [ ]:
sixPointDist = {1: 7/48, 2: 5/48, 3: 1/8, 4: 1/16, 5: 1/4, 6: 5/16}

x_vals = np.array(list(sixPointDist.keys()))
p_vals = np.array(list(sixPointDist.values()))

In [ ]:
def crude_method(x, p, n):
    F = np.cumsum(p)
    U = np.random.uniform(size=n)
    indices = np.searchsorted(F, U, side="left") #search for F(x_i-1) < U < F(x_i)
    return np.array(x)[indices]

In [ ]:
np.random.seed(42)

samples_6 = crude_method(x_vals, p_vals, n=10_000)

emp_pmf_6 = np.array([np.mean(samples_6 == xi) for xi in x_vals])

fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
ax.bar(x_vals - width/2, emp_pmf_6, width=width, alpha=0.7, label="Simulated")
ax.bar(x_vals + width/2, p_vals,    width=width, alpha=0.7, label="Theoretical")
ax.set_xticks(x_vals)
ax.set_xlabel("x")
ax.set_ylabel("Probability")
ax.set_title("Six-point distribution — simulated vs theoretical")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'x':>4} {'Theoretical':>12} {'Simulated':>10}")
for xi, pt, ps in zip(x_vals, p_vals, emp_pmf_6):
    print(f"{xi:>4} {pt:>12.4f} {ps:>10.4f}")


In [ ]:
# Rejection Method
def simple_rejection_method(x, p, n):
    k = len(x)
    c = np.max(p)   # optimal: minimises expected  number of trials
    samples = []
    while len(samples) < n:
        I  = int(np.floor(k * np.random.uniform()))  # uniform index in {0,...,k-1}
        U2 = np.random.uniform()
        if U2 <= p[I] / c:
            samples.append(x[I])
    return np.array(samples)


np.random.seed(42)
samples_rej = simple_rejection_method(x_vals, p_vals, n=10_000)
emp_pmf_rej = np.array([np.mean(samples_rej == xi) for xi in x_vals])

fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
ax.bar(x_vals - width/2, emp_pmf_rej, width=width, alpha=0.7, label="Rejection samples")
ax.bar(x_vals + width/2, p_vals,      width=width, alpha=0.7, label="Theoretical")
ax.set_xticks(x_vals)
ax.set_xlabel("x")
ax.set_ylabel("Probability")
ax.set_title("Rejection method — six-point distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Expected trials per sample: {len(x_vals) * np.max(p_vals):.3f}")


In [ ]:
# general rejection
def rejection_method(p, q, sample_q, n):
    """
    p, q     : dicts {x: probability} for target and proposal pmfs (same support)
    sample_q : callable, returns one sample x ~ q
    n        : number of accepted samples to generate
    """
    c = max(p[x] / q[x] for x in p)

    samples = []
    while len(samples) < n:
        Y = sample_q()
        U = np.random.uniform()
        if U <= p[Y] / (c * q[Y]):
            samples.append(Y)
    return np.array(samples), c


# Demo: target = sixPointDist, proposal = uniform over the same support
np.random.seed(42)

qSixPointDist = {x: 1 / len(sixPointDist) for x in sixPointDist}

def sample_q():
    return np.random.choice(list(qSixPointDist.keys()), p=list(qSixPointDist.values()))

samples_genrej, c = rejection_method(sixPointDist, qSixPointDist, sample_q, n=10_000)
emp_pmf_genrej = np.array([np.mean(samples_genrej == xi) for xi in x_vals])

fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
ax.bar(x_vals - width/2, emp_pmf_genrej, width=width, alpha=0.7, label="General rejection samples")
ax.bar(x_vals + width/2, p_vals,         width=width, alpha=0.7, label="Theoretical")
ax.set_xticks(x_vals)
ax.set_xlabel("x")
ax.set_ylabel("Probability")
ax.set_title("General rejection method — six-point distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"c = max(p_i/q_i) = {c:.4f}  (expected trials per sample)")

In [ ]:
# Alias Method

def alias_setup(p):
    n = len(p)
    L = list(range(n))                           # step 1: L = {0,...,n-1}
    F = list(np.array(p, dtype=float) * n)       # step 2: F = k*p

    G = [i for i, q in enumerate(F) if q >= 1]   # step 3
    S = [i for i, q in enumerate(F) if q <= 1]   # step 3

    while S:                           # step 4: while S not empty
        i = G[0]                       # 4a
        j = S[0]                       # 4a
        L[j] = i                       # 4b
        F[i] = F[i] - (1 - F[j])       # 4b
        if F[i] < 1 - 1e-9:            # 4c
            G.pop(0)
            S.append(i)
        S.pop(0)                       # 4d

    return np.clip(F, 0, 1), np.array(L, dtype=int)


def alias_sample(x, F, L, n):
    k = len(x)
    U1 = np.random.uniform(size=n)
    U2 = np.random.uniform(size=n)
    I  = np.floor(k * U1).astype(int)      # step 1: I = floor(k*U1), 0-indexed
    picks = np.where(U2 <= F[I], I, L[I])  # step 2: if U2 <= F(I) → I, else L(I)
    return np.array(x)[picks]

In [ ]:
np.random.seed(42)

F_table, L_table = alias_setup(p_vals)

print("Alias tables:")
print(f"{'i':>4} {'x_i':>6} {'F[i]':>10} {'L[i]':>10}")
for i, (xi, f, l) in enumerate(zip(x_vals, F_table, L_table)):
    print(f"{i:>4} {xi:>6} {f:>10.4f} {x_vals[l]:>10}")

samples_alias = alias_sample(x_vals, F_table, L_table, n=10_000)
emp_pmf_alias = np.array([np.mean(samples_alias == xi) for xi in x_vals])

fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
ax.bar(x_vals - width/2, emp_pmf_alias, width=width, alpha=0.7, label="Alias samples")
ax.bar(x_vals + width/2, p_vals,        width=width, alpha=0.7, label="Theoretical")
ax.set_xticks(x_vals)
ax.set_xlabel("x")
ax.set_ylabel("Probability")
ax.set_title("Alias method — six-point distribution")
ax.legend()
plt.tight_layout()
plt.show()


## Part 3 - Performance Comparison

In [ ]:
import time
from scipy.stats import chi2

N    = 10000
REPS = 50
np.random.seed(0)

F_table, L_table = alias_setup(p_vals)

def time_method(fn, reps=REPS):
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t0) / reps * 1e3  # ms

np.random.seed(42)
s_crude  = crude_method(x_vals, p_vals, N)
s_reject = simple_rejection_method(x_vals, p_vals, N)
s_alias  = alias_sample(x_vals, F_table, L_table, N)

t_crude  = time_method(lambda: crude_method(x_vals, p_vals, N))
t_reject = time_method(lambda: simple_rejection_method(x_vals, p_vals, N))
t_alias  = time_method(lambda: alias_sample(x_vals, F_table, L_table, N))

def chi2_result(samples):
    obs = np.array([np.sum(samples == xi) for xi in x_vals])
    stat, pval = chisquare(obs, p_vals * N)
    return stat, pval

chi_crude  = chi2_result(s_crude)
chi_reject = chi2_result(s_reject)
chi_alias  = chi2_result(s_alias)

print(f"{'Method':<20} {'Time (ms)':>10} {'χ² stat':>10} {'p-value':>10}  {'Reject H0?':>12}")
print("-" * 68)
for name, t, (stat, pval) in [
    ("Crude",     t_crude,  chi_crude),
    ("Rejection", t_reject, chi_reject),
    ("Alias",     t_alias,  chi_alias),
]:
    reject = "Yes" if pval < 0.05 else "No"
    print(f"{name:<20} {t:>10.2f} {stat:>10.3f} {pval:>10.4f}  {reject:>12}")
    
crit_val = chi2.ppf(1 - 0.05, df=len(x_vals)-1)
print(f"\nχ² critical value (df={len(x_vals)-1}, α=0.05): {crit_val:.3f}")
print(f"H0: samples follow the theoretical distribution.")
print(f"Fail to reject H0 (p > 0.05) → method produces the correct distribution.")

k = len(x_vals)
c = np.max(p_vals)
print(f"\nRejection — expected trials per sample: k·c = {k}·{c:.4f} = {k*c:.3f}")


## Exercise 3

## Part 1

### Exponential Distribution

In [ ]:
random.seed(42)
np.random.seed(42)
def simulate_exponential(lam, n):
    U = np.random.random(n)          # n uniforms
    X = -np.log(U) / lam           # inversion
    return X

exp_samples = simulate_exponential(lam=2, n=10000)

### Normal Distribution (at least with standard Box-Mueller)

In [ ]:
random.seed(42)
np.random.seed(42)

def box_muller_normal(n):
    U1 = np.random.random(n // 2)
    U2 = np.random.random(n // 2)

    # Box-Muller transformation
    R = np.sqrt(-2 * np.log(U1))
    Z1 = R * np.cos(2 * np.pi * U2)
    Z2 = R * np.sin(2 * np.pi * U2)

    Z = np.concatenate([Z1, Z2])

    # Hvis n er ulige, generer en ekstra normal variabel
    if n % 2 == 1:
        U1 = np.random.random()
        U2 = np.random.random()
        R = np.sqrt(-2 * np.log(U1))
        extra = R * np.cos(2 * np.pi * U2)
        Z = np.append(Z, extra)

    return Z

def box_muller_general(n, mu, sigma):
    Z = box_muller_normal(n)
    return mu + sigma * Z


In [ ]:
random.seed(42)
np.random.seed(42)

box_muller_samples = box_muller_general(n=10000, mu=0, sigma=1)

### Pareto Distribution

In [ ]:
random.seed(42)
np.random.seed(42)

def simulate_pareto(n, k, beta=1.0):
    U = np.random.random(n)
    X = beta * (U ** (-1.0 / k))
    return X

In [ ]:
random.seed(42)
np.random.seed(42)

ks = [2.05, 2.5, 3, 4]
pareto_samples = {k: simulate_pareto(10000, k) for k in ks}

### Sammenlign

In [ ]:
random.seed(42)
np.random.seed(42)

def theoretical_moments(dist, params):
    if dist == "exponential":
        lam = params["lam"]
        mean = 1 / lam
        var = 1 / (lam**2)
        return mean, var

    if dist == "normal":
        mu = params["mu"]
        sigma = params["sigma"]
        mean = mu
        var = sigma**2
        return mean, var

    if dist == "pareto":
        k = params["k"]
        beta = params["beta"]
        mean = beta * k / (k - 1) if k > 1 else np.nan
        var = beta**2 * k / ((k-1)**2 * (k-2)) if k > 2 else np.nan
        return mean, var

    raise ValueError(f"Unknown distribution: {dist}")


def theoretical_cdf(x, dist, params):
    x = np.asarray(x)

    if dist == "exponential":
        lam = params["lam"]
        return np.where(x < 0, 0.0, 1 - np.exp(-lam * x))

    if dist == "normal":
        mu = params["mu"]
        sigma = params["sigma"]
        return stats.norm.cdf(x, loc=mu, scale=sigma)

    if dist == "pareto":
        k = params["k"]
        beta = params["beta"]
        return np.where(x < beta, 0.0, 1 - (beta / x) ** k)

    raise ValueError(f"Unknown distribution: {dist}")


def verify_distribution(samples, dist, params):
    emp_mean = np.mean(samples)
    emp_var = np.var(samples)

    th_mean, th_var = theoretical_moments(dist, params)
    ks_stat, ks_pvalue = stats.kstest(
        samples,
        lambda x: theoretical_cdf(x, dist, params)
    )

    print("Empirical mean:", emp_mean)
    print("Theoretical mean:", th_mean)
    print()
    print("Empirical variance:", emp_var)
    print("Theoretical variance:", th_var)
    print()
    print("KS statistic:", ks_stat)
    print("KS p-value:", ks_pvalue)


def plot_distribution_axis(ax, samples, dist, params, bins=100, title=None):
    if dist == "exponential":
        lam = params["lam"]
        xs = np.linspace(0, np.max(samples), 500)
        pdf = lam * np.exp(-lam * xs)

    elif dist == "normal":
        mu = params["mu"]
        sigma = params["sigma"]
        xs = np.linspace(np.min(samples), np.max(samples), 500)
        pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((xs - mu) / sigma) ** 2)

    elif dist == "pareto":
        k = params["k"]
        beta = params["beta"]
        xs = np.linspace(beta, np.max(samples), 500)
        pdf = np.where(xs >= beta, (k * beta**k) / (xs ** (k + 1)), 0.0)

    else:
        raise ValueError(f"Unknown distribution: {dist}")

    ax.hist(samples, bins=bins, density=True, alpha=0.5, label="Empirical")
    ax.plot(xs, pdf, "r", label="Theoretical PDF")
    if title is not None:
        ax.set_title(title)
    ax.legend()


def plot_distribution(samples, dist, params, bins=100):
    fig, ax = plt.subplots(figsize=(7, 4))
    plot_distribution_axis(ax, samples, dist, params, bins=bins)
    plt.tight_layout()
    plt.show()


In [ ]:
random.seed(42)
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_distribution_axis(axes[0], exp_samples, "exponential", {"lam": 2}, title="Exponential distribution, $\lambda=2$")
plot_distribution_axis(axes[1], box_muller_samples, "normal", {"mu": 0, "sigma": 1}, title="Normal distribution, $\mu=0$, $\sigma=1$")
plt.tight_layout()
plt.show()

verify_distribution(exp_samples, "exponential", {"lam": 2})
print()
verify_distribution(box_muller_samples, "normal", {"mu": 0, "sigma": 1})

In [ ]:
random.seed(42)
np.random.seed(42)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, k in zip(axes, ks):
    plot_distribution_axis(ax, pareto_samples[k], "pareto", {"k": k, "beta": 1}, title=f"Pareto distribution, $beta=1$, $k={k}$")
    ax.set_xlim(0,10)
    ax.set_ylim(0, 5)

plt.tight_layout()
plt.show()

for k in ks:
    verify_distribution(pareto_samples[k], "pareto", {"k": k, "beta": 1})
    print()

In [ ]:
random.seed(42)
np.random.seed(42)

# Part 1: Combined comparison plot and KS-tests
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
samples_list = [exp_samples, box_muller_samples, pareto_samples[2.5]]
dists = ["exponential", "normal", "pareto"]
params_list = [{"lam": 2}, {"mu": 0, "sigma": 1}, {"k": 2.5, "beta": 1}]
titles = ["Exponential (\u03bb=2)", "Normal (0,1)", "Pareto (k=2.5, \u03b2=1)"]

for ax, samples, dist, params, title in zip(axes, samples_list, dists, params_list, titles):
    plot_distribution_axis(ax, samples, dist, params, bins=100, title=title)
    # clip x-limits using central percentiles so heavy tails don't collapse the plot
    low = np.percentile(samples, 0.5)
    high = np.percentile(samples, 99.5)
    ax.set_xlim(max(low, np.min(samples)), high)

plt.tight_layout()
plt.show()

print("Goodness-of-fit (KS test) results for Part 1:")
for dist, samples, params in zip(dists, samples_list, params_list):
    ks_stat, ks_p = stats.kstest(samples, lambda x: theoretical_cdf(x, dist, params))
    print(f"{dist}: KS stat = {ks_stat:.4f}, p-value = {ks_p:.4f}")

In [ ]:
random.seed(42)
np.random.seed(42)

# Part 1 extended: Exponential, Normal and all Pareto(k) side-by-side (2x3 grid)
fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharey=True)
axes = axes.ravel()

# prepare samples and parameters
pareto_order = ks  # [2.05, 2.5, 3, 4]
samples_list = [exp_samples, box_muller_samples] + [pareto_samples[k] for k in pareto_order]
dists = ["exponential", "normal"] + ["pareto"] * len(pareto_order)
params_list = [{"lam": 2}, {"mu": 0, "sigma": 1}] + [{"k": k, "beta": 1} for k in pareto_order]
titles = ["Exponential (\u03bb=2)", "Normal (0,1)"] + [f"Pareto (k={k}, \u03b2=1)" for k in pareto_order]

def get_scipy_rv(dist, params):
    if dist == "exponential":
        lam = params["lam"]
        return stats.expon(scale=1/lam)
    if dist == "normal":
        return stats.norm(loc=params["mu"], scale=params["sigma"])
    if dist == "pareto":
        k = params["k"]
        beta = params.get("beta", 1)
        return stats.pareto(b=k, scale=beta)
    raise ValueError(f"Unknown dist: {dist}")

def chi2_equal_prob_bins(samples, dist, params, bins=20):
    rv = get_scipy_rv(dist, params)
    n = len(samples)
    qs = np.linspace(0, 1, bins + 1)

    edges = rv.ppf(qs)
    edges[0] = min(np.min(samples), edges[0]) if np.isfinite(edges[0]) else np.min(samples)
    edges[-1] = np.inf
    counts, _ = np.histogram(samples, bins=edges)
    expected = np.ones_like(counts) * (n / bins)
    chi_stat, p_value = stats.chisquare(f_obs=counts, f_exp=expected)
    return chi_stat, p_value, counts, expected, edges

for ax, samples, dist, params, title in zip(axes, samples_list, dists, params_list, titles):
    plot_distribution_axis(ax, samples, dist, params, bins=100, title=title)
    if dist == "pareto":
        low = params.get("beta", np.min(samples))
        high = np.percentile(samples, 99.5)
        ax.set_xlim(low, high)
    else:
        low = np.percentile(samples, 0.5)
        high = np.percentile(samples, 99.5)
        ax.set_xlim(max(low, np.min(samples)), high)

for i in range(len(samples_list), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("Goodness-of-fit (KS and Chi-squared tests) results for Part 1 (including all Pareto k):")
for dist, samples, params in zip(dists, samples_list, params_list):
    ks_stat, ks_p = stats.kstest(samples, lambda x: theoretical_cdf(x, dist, params))
    chi_stat, chi_p, counts, expected, edges = chi2_equal_prob_bins(samples, dist, params, bins=20)
    print(f"{dist} {params}: KS stat = {ks_stat:.4f}, p-value = {ks_p:.4f}")
    print(f"{dist} {params}: Chi-squared stat = {chi_stat:.4f}, p-value = {chi_p:.4f}")

## Part 2

In [ ]:
random.seed(42)
np.random.seed(42)

def estimate_probability(samples, condition):
    indicators = np.array([condition(x) for x in samples])
    return np.mean(indicators)

def estimate_expectation(samples, g):
    values = np.array([g(x) for x in samples])
    return np.mean(values)

def confidence_interval(samples, alpha=0.05):
    mean = np.mean(samples)
    std = np.std(samples, ddof=1)
    n = len(samples)
    z = 1.96  # for 95%
    return mean - z*std/np.sqrt(n), mean + z*std/np.sqrt(n)

In [ ]:
random.seed(42)
np.random.seed(42)

X = simulate_pareto(100000, k=2.5, beta=1)

# P(X > 10)
p = estimate_probability(X, lambda x: x > 10)

# E[X]
m = estimate_expectation(X, lambda x: x)

# CI for E[X]
ci = confidence_interval(X)


In [ ]:
random.seed(42)
np.random.seed(42)

verify_distribution(pareto_samples[2.5], "pareto", {"k": 2.5, "beta": 1})
print("P(X > 10):", p)
print("E[X]:", m)
print("CI for E[X]:", ci)
plot_distribution(pareto_samples[2.5], "pareto", {"k": 2.5, "beta": 1})

## Part 3

In [ ]:
random.seed(42)
np.random.seed(42)

def ci_mean_and_var_normal(mu_true=0.0, sigma2_true=1.0, n=10, n_rep=100):
    t_975 = 2.262      # t_{0.975, 9}
    chi2_975 = 19.02   # chi^2_{0.975, 9}
    chi2_025 = 2.70    # chi^2_{0.025, 9}

    mean_intervals = []
    var_intervals = []
    mean_contains = 0
    var_contains = 0

    for _ in range(n_rep):
        x = box_muller_normal(n)  # N(0,1)

        # sample mean and variance
        x_bar = np.mean(x)
        s2 = np.var(x, ddof=1)

        # 95% CI for mean (unknown variance → t)
        half_width_mean = t_975 * np.sqrt(s2 / n)
        ci_mean = (x_bar - half_width_mean, x_bar + half_width_mean)

        # 95% CI for variance (chi-square)
        df = n - 1
        ci_var = (df * s2 / chi2_975, df * s2 / chi2_025)

        mean_intervals.append(ci_mean)
        var_intervals.append(ci_var)

        if ci_mean[0] <= mu_true <= ci_mean[1]:
            mean_contains += 1
        if ci_var[0] <= sigma2_true <= ci_var[1]:
            var_contains += 1

    return mean_intervals, var_intervals, mean_contains, var_contains

In [ ]:
random.seed(42)
np.random.seed(42)

mean_ints, var_ints, mean_hits, var_hits = ci_mean_and_var_normal()

print("Mean CIs containing true mean:", mean_hits, "ud af 100")
print("Var CIs containing true variance:", var_hits, "ud af 100")


## Part 4

In [ ]:
random.seed(42)
np.random.seed(42)

def sample_pareto_composition(beta, k, n, rng):
    lam = rng.gamma(shape=k, scale=1.0 / beta, size=n)  # λ ~ Gamma(k, rate=β)
    X   = rng.exponential(scale=1.0 / lam)               # X|λ ~ Exp(λ)
    return X + beta 

In [ ]:
random.seed(42)
np.random.seed(42)

X1 = simulate_pareto(100000, k=4)        
X2 = sample_pareto_composition(beta=1, k=4, n=100000, rng=np.random)

print(np.mean(X1), np.mean(X2))
print(np.var(X1), np.var(X2))

In [ ]:
random.seed(42)
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)
axes = axes.ravel()

k = 4

X1 = simulate_pareto(100000, k=k)        
X2 = sample_pareto_composition(beta=1, k=k, n=100000, rng=np.random)

# Plotting the two Pareto(k) samples side by side
plot_distribution_axis(axes[0], X1, "pareto", {"k": k, "beta": 1}, title=f"Pareto({k}) - Direct Inversion Method")
plot_distribution_axis(axes[1], X2, "pareto", {"k": k, "beta": 1}, title=f"Pareto({k}) - Composition Method")
axes[0].set_xlim(1, 6)
# axes[0].set_ylim(0, 5)
axes[1].set_xlim(1, 6)
# axes[1].set_ylim(0, 5)

# plot the theoretical PDF for Pareto(k)
xs = np.linspace(1, 5, 500)
pdf = k * (1 ** k) / (xs ** (k + 1))
axes[0].plot(xs, pdf, "r", label="Theoretical PDF")
axes[1].plot(xs, pdf, "r", label="Theoretical PDF")

# do a KS test to verify both samples match the theoretical distribution
ks_stat1, ks_p1 = stats.kstest(X1, lambda x: theoretical_cdf(x, "pareto", {"k": k, "beta": 1}))
ks_stat2, ks_p2 = stats.kstest(X2, lambda x: theoretical_cdf(x, "pareto", {"k": k, "beta": 1}))
print(f"Direct Inversion Method: KS stat = {ks_stat1:.4f}, p-value = {ks_p1:.4f}")
print(f"Composition Method: KS stat = {ks_stat2:.4f}, p-value = {ks_p2:.4f}")

# do a chi-squared test with equal-probability bins
def chi2_equal_prob_bins(samples, dist, params, bins=20):
    rv = get_scipy_rv(dist, params)
    n = len(samples)
    qs = np.linspace(0, 1, bins + 1)
    edges = rv.ppf(qs)
    edges[0] = min(np.min(samples), edges[0]) if np.isfinite(edges[0]) else np.min(samples)
    edges[-1] = np.inf
    counts, _ = np.histogram(samples, bins=edges)
    expected = np.ones_like(counts) * (n / bins)
    chi_stat, p_value = stats.chisquare(f_obs=counts, f_exp=expected)
    return chi_stat, p_value

chi_stat1, chi_p1 = chi2_equal_prob_bins(X1, "pareto", {"k": k, "beta": 1}, bins=20)
chi_stat2, chi_p2 = chi2_equal_prob_bins(X2, "pareto", {"k": k, "beta": 1}, bins=20)
print(f"Direct Inversion Method: Chi-squared stat = {chi_stat1:.4f}, p-value = {chi_p1:.4f}")
print(f"Composition Method: Chi-squared stat = {chi_stat2:.4f}, p-value = {chi_p2:.4f}")

# ttest for difference in means
t_stat, t_p = stats.ttest_ind(X1, X2, equal_var=False)
print(f"T-test for difference in means: t-statistic = {t_stat:.4f}, p-value = {t_p:.4f}")

## Exercise 4

## Part 1

In [ ]:
def erlang_b(A, m):
    num = (A**m) / factorial(m)
    den = sum([(A**i) / factorial(i) for i in range(m + 1)])
    return num / den

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h

In [ ]:
def sim_block_system(m, num_customers, arrival_gen, service_gen):
    clock = 0.0
    blocked_count = 0
    # priority kø
    servers = [] 
    
    for _ in range(num_customers):
        clock += arrival_gen()
        
        while servers and servers[0] <= clock:
            heapq.heappop(servers)
            
        if len(servers) < m:
            service_time = service_gen()
            heapq.heappush(servers, clock + service_time)
        else:
            blocked_count += 1
            
    return blocked_count / num_customers

In [ ]:
np.random.seed(42)
m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000
num_runs = 10

arrival_gen = lambda: np.random.exponential(mean_interarrival)
service_gen = lambda: np.random.exponential(mean_service)

results_p1 = [sim_block_system(m, num_customers, arrival_gen, service_gen) for _ in range(num_runs)]
mean_p1, ci_low_p1, ci_high_p1 = mean_confidence_interval(results_p1)
exact_p1 = erlang_b(8.0, 10)

print(f"  Observed Blocked Fraction: {mean_p1:.4f} (95% CI: [{ci_low_p1:.4f}, {ci_high_p1:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")

## Part 2

### a - Erlang arrivals

In [ ]:
np.random.seed(42)  

k_erlang = 2
erlang_arrival = lambda: np.random.gamma(k_erlang, 1.0/k_erlang)

results_p2a = [sim_block_system(m, num_customers, erlang_arrival, service_gen) for _ in range(num_runs)]
mean_p2a, ci_low_p2a, ci_high_p2a = mean_confidence_interval(results_p2a)

print(f"  Observed Blocked Fraction: {mean_p2a:.4f} (95% CI: [{ci_low_p2a:.4f}, {ci_high_p2a:.4f}])")

### b - Hyperexponential arrivals

In [ ]:
def hyperexponential_arrival():
    if np.random.rand() < 0.8:
        return np.random.exponential(1/0.8333)
    else:
        return np.random.exponential(1/5.0)

In [ ]:
results_p2b = [sim_block_system(m, num_customers, hyperexponential_arrival, service_gen) for _ in range(num_runs)]
mean_p2b, ci_low_p2b, ci_high_p2b = mean_confidence_interval(results_p2b)

print(f"  Observed Blocked Fraction: {mean_p2b:.4f} (95% CI: [{ci_low_p2b:.4f}, {ci_high_p2b:.4f}])")

## Part 3

### a - constant service time

In [ ]:
np.random.seed(42)
constant_service = lambda: 8.0

results_p3a = [sim_block_system(m, num_customers, arrival_gen, constant_service) for _ in range(num_runs)]
mean_p3a, ci_low_p3a, ci_high_p3a = mean_confidence_interval(results_p3a)

print(f"  Observed Blocked Fraction: {mean_p3a:.4f} (95% CI: [{ci_low_p3a:.4f}, {ci_high_p3a:.4f}])")

### b - Pareto distributed service time

In [ ]:
np.random.seed(42)

def pareto_service(k, mean_target=8.0):
    beta = mean_target * (k - 1) / k
    return (np.random.pareto(k) + 1) * beta

pareto_gen = lambda: pareto_service(2.05)
results_p3b = [sim_block_system(m, num_customers, arrival_gen, pareto_gen) for _ in range(num_runs)]
mean_p3b, ci_low_p3b, ci_high_p3b = mean_confidence_interval(results_p3b)

print(f"  Observed Blocked Fraction: {mean_p3b:.4f} (95% CI: [{ci_low_p3b:.4f}, {ci_high_p3b:.4f}])")

### c - choose one or two other distributions

In [ ]:
np.random.seed(42)

def gamma_service(shape=3.0, mean_target=8.0):
    scale = mean_target / shape
    return np.random.gamma(shape, scale)


def lognormal_service(sigma=0.6, mean_target=8.0):
    mu = np.log(mean_target) - 0.5 * sigma**2
    return np.random.lognormal(mean=mu, sigma=sigma)

service_gamma = lambda: gamma_service(shape=3.0, mean_target=8.0)
service_lognormal = lambda: lognormal_service(sigma=0.6, mean_target=8.0)

results_p3c_gamma = [sim_block_system(m, num_customers, arrival_gen, service_gamma) for _ in range(num_runs)]
mean_p3c_gamma, ci_low_p3c_gamma, ci_high_p3c_gamma = mean_confidence_interval(results_p3c_gamma)

print(f"Gamma service:     Observed Blocked Fraction: {mean_p3c_gamma:.4f} (95% CI: [{ci_low_p3c_gamma:.4f}, {ci_high_p3c_gamma:.4f}])")

results_p3c_lognormal = [sim_block_system(m, num_customers, arrival_gen, service_lognormal) for _ in range(num_runs)]
mean_p3c_lognormal, ci_low_p3c_lognormal, ci_high_p3c_lognormal = mean_confidence_interval(results_p3c_lognormal)

print(f"Lognormal service: Observed Blocked Fraction: {mean_p3c_lognormal:.4f} (95% CI: [{ci_low_p3c_lognormal:.4f}, {ci_high_p3c_lognormal:.4f}])")

## Part 5

In [ ]:
labels = [
    "Part 1\nExp/Exp",
    "Part 2a\nErlang arrivals",
    "Part 2b\nHyperexp arrivals",
    "Part 3a\nConstant service",
    "Part 3b\nPareto service",
    "Part 3c\nGamma service",
    "Part 3c\nLognormal service",
]

means = [
    mean_p1,
    mean_p2a,
    mean_p2b,
    mean_p3a,
    mean_p3b,
    mean_p3c_gamma,
    mean_p3c_lognormal,
]
lows = [
    mean_p1 - ci_low_p1,
    mean_p2a - ci_low_p2a,
    mean_p2b - ci_low_p2b,
    mean_p3a - ci_low_p3a,
    mean_p3b - ci_low_p3b,
    mean_p3c_gamma - ci_low_p3c_gamma,
    mean_p3c_lognormal - ci_low_p3c_lognormal,
]
highs = [
    ci_high_p1 - mean_p1,
    ci_high_p2a - mean_p2a,
    ci_high_p2b - mean_p2b,
    ci_high_p3a - mean_p3a,
    ci_high_p3b - mean_p3b,
    ci_high_p3c_gamma - mean_p3c_gamma,
    ci_high_p3c_lognormal - mean_p3c_lognormal,
]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
ax.bar(x, means, yerr=[lows, highs], capsize=5, color=["#4C78A8"] * len(labels), alpha=0.85)
ax.axhline(exact_p1, color="#FF5733", linestyle="--", label="Exact Erlang B")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Blocked fraction")
ax.set_title("Comparison of distributions with 95% confidence intervals")
ax.set_ylim(0, max(max(highs) + max(means), exact_p1) * 1.25)
ci_handle = Line2D([0], [0], color="#4C78A8", linewidth=2, marker="|", markersize=12, label="95% CI")
exact_handle = Line2D([0], [0], color="#FF5733", linestyle="--", linewidth=2, label="Exact Erlang B")
ax.legend(handles=[ci_handle, exact_handle])
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Exercise 5

In [ ]:
def confidence(est,alpha, data, n):
    z = stats.norm.ppf(1 - alpha / 2)
    std = np.std(data, ddof=1)
    error = z * std / np.sqrt(n)
    lower = est - error
    upper = est + error
    print(f"{(1 - alpha) * 100:.1f}% confidence interval: \n[{lower:.4f}, {upper:.4f}]")


## Part 1

In [ ]:
# Monte Carlo estimator
np.random.seed(42)
n_mc = 100
Ui_mc = np.random.uniform(0, 1, n_mc)
Xi = np.exp(Ui_mc)
est_mc = 1/n_mc * np.sum(Xi)
print(f"Monte Carlo estimator: {est_mc:.4f}")

# Konfidens intervaller
alpha_mc = 0.05
confidence(est_mc, alpha_mc, Xi, n_mc)

## Part 2

In [ ]:
# Antithetic variables
np.random.seed(42)
n_av = 100
Ui_av = np.random.uniform(0, 1, n_av)
Xi_av = np.exp(Ui_av)
Yi_av = (np.exp(Ui_av)+np.exp(1 - Ui_av)) / 2
est_av = 1/n_av * np.sum(Yi_av)
print(f"Antithetic variables estimator: {est_av:.4f}")

# Konfidens interval
alpha_av = 0.05
confidence(est_av, alpha_av, Yi_av, n_av)

# Part 3

In [ ]:
# control variates
np.random.seed(42)
n_cv = 100
Ui_cv = np.random.uniform(0, 1, n_cv)
Xi_cv = np.exp(Ui_cv)
Zi = Ui_cv
mu_Z = 0.5
c = -np.cov(Xi_cv, Zi)[0, 1] / np.var(Zi)
Yi_cv = Xi_cv + c * (Zi - mu_Z)
est_cv = 1/n_cv * np.sum(Yi_cv)
print(f"Control variates estimator: {est_cv:.4f}")  

# Konfidens interval
alpha_cv = 0.05
confidence(est_cv, alpha_cv, Yi_cv, n_cv)

## Part 4

In [ ]:
# Stratified sampling
np.random.seed(42)
n_ss = 100
strata = 10
m = n_ss // strata

Uss = np.zeros((m, strata))

for j in range(1, strata+1):
    a = (j-1)/strata
    b = j/strata
    Uss[:, j-1] = np.random.uniform(a, b, m)

Xi_ss = np.exp(Uss)
Yi_ss = 1/strata * np.sum(Xi_ss, axis=1)

est_ss = 1/m * np.sum(Yi_ss)
print(f"Stratified sampling estimator: {est_ss:.4f}")

# Konfidens interval
alpha_ss = 0.05
confidence(est_ss, alpha_ss, Yi_ss, m)

## Part 5

In [ ]:
def sim_block_system(m, num_customers, arrival_gen, service_gen):
    clock = 0.0
    blocked_count = 0
    # priority kø
    servers = [] 
    
    for _ in range(num_customers):
        clock += arrival_gen()
        
        while servers and servers[0] <= clock:
            heapq.heappop(servers)
            
        if len(servers) < m:
            service_time = service_gen()
            heapq.heappush(servers, clock + service_time)
        else:
            blocked_count += 1
            
    return blocked_count / num_customers

In [ ]:
# Control variates for blocking probability
np.random.seed(42)
n_pcv = 100   # number of simulation runs

m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000

arrival_gen = lambda: np.random.exponential(mean_interarrival)

Xi_pcv = np.zeros(n_pcv)   # X_i = blocking fraction
Zi_pcv = np.zeros(n_pcv)      # Z_i = control variate

for i in range(n_pcv):
    total_service = [0.0]   # list so it can be mutated inside function

    def service_gen_pcv():
        s = np.random.exponential(mean_service)
        total_service[0] += s
        return s

    Xi_pcv[i] = sim_block_system(m, num_customers, arrival_gen, service_gen_pcv)

    # FIX: extract the float, not the list
    Zi_pcv[i] = total_service[0]

# mean of Z
mu_Z_pcv = np.mean(Zi_pcv)

# c = -Cov(X,Z) / Var(Z)
c_pcv = -np.cov(Xi_pcv, Zi_pcv, ddof=1)[0, 1] / np.var(Zi_pcv, ddof=1)

# Y_i = X_i + c (Z_i - mu_Z)
Yi_pcv = Xi_pcv + c_pcv * (Zi_pcv - mu_Z_pcv)

# estimator
est_pcv = np.mean(Yi_pcv)
print(f"Control variates estimator: {est_pcv:.4f}")

# Konfidens interval
alpha_pcv = 0.05
confidence(est_pcv, alpha_pcv, Yi_pcv, n_pcv)

## Part 6

In [ ]:
np.random.seed(42)

m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000
num_runs = 10

p = 0.8
mean_fast = 1/0.8333 
mean_slow = 1/5.0     

service_gen = lambda: np.random.exponential(mean_service)

# uden CRN
arrival_gen_poisson = lambda: np.random.exponential(mean_interarrival)

def hyperexponential_arrival():
    if np.random.rand() < p:
        return np.random.exponential(mean_fast)
    else:
        return np.random.exponential(mean_slow)


# med CRN 

U1 = np.random.rand(num_customers)   # mixture + Poisson
U2 = np.random.rand(num_customers)   # exponential sampling i hyperexp

def poisson_arrivals_CRN():
    i = 0
    def gen():
        nonlocal i
        x = -mean_interarrival * np.log(1 - U1[i])
        i += 1
        return x
    return gen

def hyperexp_arrivals_CRN():
    i = 0
    def gen():
        nonlocal i
        if U1[i] < p:
            x = -mean_fast * np.log(1 - U2[i])
        else:
            x = -mean_slow * np.log(1 - U2[i])
        i += 1
        return x
    return gen


# uden CRN

results_p1 = [
    sim_block_system(m, num_customers, arrival_gen_poisson, service_gen)
    for _ in range(num_runs)
]

results_p2 = [
    sim_block_system(m, num_customers, hyperexponential_arrival, service_gen)
    for _ in range(num_runs)
]

# med CRN
results_p1_CRN = [
    sim_block_system(m, num_customers, poisson_arrivals_CRN(), service_gen)
    for _ in range(num_runs)
]

results_p2_CRN = [
    sim_block_system(m, num_customers, hyperexp_arrivals_CRN(), service_gen)
    for _ in range(num_runs)
]


# varians sammenligning
diff_noCRN = np.array(results_p1) - np.array(results_p2)
diff_CRN   = np.array(results_p1_CRN) - np.array(results_p2_CRN)

alpha = 0.05
print("\n--- UDEN CRN ---")
print("Poisson mean:", np.mean(results_p1))
print("HyperExp mean:", np.mean(results_p2))
print("Var(diff):", np.var(diff_noCRN, ddof=1))

print("\n--- MED CRN ---")
print("Poisson mean:", np.mean(results_p1_CRN))
print("HyperExp mean:", np.mean(results_p2_CRN))
print("Var(diff):", np.var(diff_CRN, ddof=1))

## Part 7

In [ ]:
# Crude MC estimator for Normal fordeling
def crude_mc(a, n):
    Z = np.random.randn(n)
    return np.mean(Z > a)

def importance_sampling(a, n, sigma=1.0):
    Y = np.random.normal(loc=a, scale=sigma, size=n)
    f = norm.pdf(Y, 0, 1)
    g = norm.pdf(Y, a, sigma)
    w = f / g
    return np.mean((Y > a) * w)

# --- Experiments ---
for a in [2, 4]:
    for n in [1000, 10000, 100000]:
        crude = crude_mc(a, n)
        est_is = importance_sampling(a, n, sigma=1.0)
        print(f"a={a}, n={n}: crude={crude:.4f}, estimator={est_is:.6f}")

        # Konfidens interval for crude MC
        alpha = 0.05
        print(f"Crude")
        confidence(crude, alpha, (np.random.randn(n) > a).astype(float), n)
        # Konfidens interval for IS estimator
        print(f"Importance Sampling")
        confidence(est_is, alpha, (importance_sampling(a, n, sigma=1.0) * (np.random.normal(loc=a, scale=1.0, size=n) > a)).astype(float), n)

## Part 8

In [ ]:
# h(x) = e^x
def h(x):
    return np.exp(x)

# f(x) = 1 on [0,1], 0 otherwise
def f(x):
    return (0 <= x) & (x <= 1)

# g(x) = λ e^{-λx}
def g(x, lam):
    return lam * np.exp(-lam * x)

# Importance sampling estimator using Y_i ~ g
def theta_IS(lam, Y):
    estimator = h(Y) * f(Y) / g(Y, lam)

    return np.mean(estimator)

lam = 1.0
Y = np.random.exponential(scale=1/lam, size=10000)
print(theta_IS(lam, Y))
confidence(theta_IS(lam, Y), 0.05, (h(Y) * f(Y) / g(Y, lam)).astype(float), len(Y))

## Exercise 6

In [ ]:
np.random.seed(42)

m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000
num_runs = 10

# Calculate Offered Traffic (A)
A = (1.0 / mean_interarrival) * mean_service # Equals 8.0

def f(x, A, m):
    """
    Target unnormalized distribution f(x).
    Proportional to the truncated Poisson probability P(x).
    """
    if 0 <= x <= m:
        return (A**x) / math.factorial(x)
    return 0.0

def g(y, x):
    """
    Proposal distribution g(y | x).
    The probability of proposing state 'y' given current state 'x'.
    For a symmetric random walk of +1 or -1, the probability is 0.5.
    """
    if y == x + 1 or y == x - 1:
        return 0.5
    return 0.0

def metropolis_hastings_erlang(m, A, num_samples, burn_in=2000, thinning=20):
    total_steps = burn_in + (num_samples * thinning)
    samples = np.zeros(num_samples, dtype=int)
    
    x = 0  # Initialize the chain at current state x = 0
    
    sample_idx = 0
    for step in range(total_steps):
        step_direction = np.random.choice([-1, 1])
        y = x + step_direction
        
        f_x = f(x, A, m)
        f_y = f(y, A, m)
        
        g_y_given_x = g(y, x)
        g_x_given_y = g(x, y)
        
        if f_x > 0 and g_y_given_x > 0:
            acceptance_ratio = (f_y * g_x_given_y) / (f_x * g_y_given_x)
            acceptance_prob = min(1.0, acceptance_ratio)
        else:
            acceptance_prob = 0.0
            
        # 5. Accept or reject
        if np.random.rand() < acceptance_prob:
            x = y
        
        if step >= burn_in and (step - burn_in) % thinning == 0:
            samples[sample_idx] = x
            sample_idx += 1
            
    return samples

total_samples = num_customers * num_runs
samples = metropolis_hastings_erlang(m, A, total_samples, burn_in=2000, thinning=1)

# Verify with a Chi-squared test
observed_counts = np.bincount(samples, minlength=m+1)

unnorm_probs = [f(i, A, m) for i in range(m + 1)]
norm_const = sum(unnorm_probs)
expected_probs = [p / norm_const for p in unnorm_probs]
expected_counts = np.array(expected_probs) * total_samples

chi2_stat, p_value = chisquare(f_obs=observed_counts, f_exp=expected_counts)

print(f"Offered Traffic (A) : {A} Erlang")
print(f"Chi-squared Stat    : {chi2_stat:.4f}")
print(f"P-value             : {p_value:.7f}")

if p_value > 0.05:
    print("Result: The Metropolis-Hastings sample matches the target distribution (Fail to reject H0).")
else:
    print("Result: The sample differs significantly from the target distribution (Reject H0).")


plt.hist(samples, bins=np.arange(-0.5, m+1.5, 1), density=True, alpha=0.6, color='g', label='M-H Samples')
plt.plot(range(m + 1), expected_probs, 'ro-', label='Expected Probabilities')
plt.title('Metropolis-Hastings Samples vs Expected Distribution')
plt.xlabel('Number of Busy Lines (i)')
plt.ylabel('Probability')
plt.xticks(range(m + 1))
plt.legend()
plt.grid()
plt.savefig('metropolis_hastings_erlang.png')

In [ ]:
import numpy as np
import math
from scipy.stats import chisquare
np.random.seed(42)
# --- 1. Parameters ---
A1 = 4.0
A2 = 4.0
m = 10
num_samples = 10000
burn_in_steps = 2000
thinning_steps = 1


P_table = np.zeros((m + 1, m + 1))
for i in range(m + 1):
    for j in range(m + 1):
        if i + j <= m:
            P_table[i, j] = (A1**i / math.factorial(i)) * (A2**j / math.factorial(j))

# The expected probabilities for the Chi-squared test
expected_probs = P_table / np.sum(P_table)


gibbs_cdfs = []
for other_val in range(m + 1):
    limit = m - other_val
    if limit < 0:
        gibbs_cdfs.append(np.array([1.0])) 
        continue
    probs = np.array([(A1**k) / math.factorial(k) for k in range(limit + 1)])
    probs /= np.sum(probs)
    gibbs_cdfs.append(np.cumsum(probs))

def mh_direct_fast(num_samples, burn_in, thinning):
    total_steps = burn_in + (num_samples * thinning)
    samples = np.zeros((num_samples, 2), dtype=int)
    i, j = 0, 0
    
    moves_i = np.array([1, -1, 0, 0])
    moves_j = np.array([0, 0, 1, -1])
    step_choices = np.random.randint(0, 4, size=total_steps)
    rand_acc = np.random.rand(total_steps)
    
    sample_idx = 0
    for step in range(total_steps):
        move_idx = step_choices[step]
        prop_i = i + moves_i[move_idx]
        prop_j = j + moves_j[move_idx]
        
        if 0 <= prop_i <= m and 0 <= prop_j <= m and (prop_i + prop_j <= m):
            p_curr = P_table[i, j]
            p_prop = P_table[prop_i, prop_j]
            
            if rand_acc[step] < (p_prop / p_curr):
                i, j = prop_i, prop_j
                
        if step >= burn_in and (step - burn_in) % thinning == 0:
            samples[sample_idx, 0] = i
            samples[sample_idx, 1] = j
            sample_idx += 1
            
    return samples

def mh_coordinate_wise_fast(num_samples, burn_in, thinning):
    total_steps = burn_in + (num_samples * thinning)
    samples = np.zeros((num_samples, 2), dtype=int)
    i, j = 0, 0
    
    moves_i = np.random.choice([-1, 1], size=total_steps)
    moves_j = np.random.choice([-1, 1], size=total_steps)
    rand_acc_i = np.random.rand(total_steps)
    rand_acc_j = np.random.rand(total_steps)
    
    sample_idx = 0
    for step in range(total_steps):
        prop_i = i + moves_i[step]
        if 0 <= prop_i <= m and (prop_i + j <= m):
            if rand_acc_i[step] < (P_table[prop_i, j] / P_table[i, j]):
                i = prop_i
                
        prop_j = j + moves_j[step]
        if 0 <= prop_j <= m and (i + prop_j <= m):
            if rand_acc_j[step] < (P_table[i, prop_j] / P_table[i, j]):
                j = prop_j
                
        if step >= burn_in and (step - burn_in) % thinning == 0:
            samples[sample_idx, 0] = i
            samples[sample_idx, 1] = j
            sample_idx += 1
            
    return samples

def gibbs_sampling_fast(num_samples, burn_in, thinning):
    total_steps = burn_in + (num_samples * thinning)
    samples = np.zeros((num_samples, 2), dtype=int)
    i, j = 0, 0
    
    rand_i = np.random.rand(total_steps)
    rand_j = np.random.rand(total_steps)
    
    sample_idx = 0
    for step in range(total_steps):
        i = np.searchsorted(gibbs_cdfs[j], rand_i[step])
        j = np.searchsorted(gibbs_cdfs[i], rand_j[step])
        
        if step >= burn_in and (step - burn_in) % thinning == 0:
            samples[sample_idx, 0] = i
            samples[sample_idx, 1] = j
            sample_idx += 1
            
    return samples

def run_verification(samples, name):
    # Count occurrences
    observed = np.zeros((m + 1, m + 1))
    for i, j in samples:
        observed[i, j] += 1
        
    obs_1d, exp_1d = [], []
    for i in range(m + 1):
        for j in range(m + 1):
            if expected_probs[i, j] > 0:
                obs_1d.append(observed[i, j])
                exp_1d.append(expected_probs[i, j] * len(samples))
                
    chi2_stat, p_value = chisquare(f_obs=obs_1d, f_exp=exp_1d)
    
    print(f"--- {name} ---")
    print(f"Chi-squared Stat : {chi2_stat:.4f}")
    print(f"P-value          : {p_value:.4e}")
    if p_value > 0.05:
        print("Result: Valid match to target distribution.\n")
    else:
        print("Result: Differs significantly from target distribution.\n")

run_verification(mh_direct_fast(num_samples, burn_in_steps, thinning_steps), "(a) Direct M-H")
run_verification(mh_coordinate_wise_fast(num_samples, burn_in_steps, thinning_steps), "(b) Coordinate-wise M-H")
run_verification(gibbs_sampling_fast(num_samples, burn_in_steps, thinning_steps), "(c) Gibbs Sampling")

In [ ]:
np.random.seed(42)
rho = 0.5
cov_matrix = np.array([[1, rho], [rho, 1]])
xi_true, gamma_true = np.random.multivariate_normal(mean=[0, 0], cov=cov_matrix)
theta_true, psi_true = np.exp(xi_true), np.exp(gamma_true)




n = 10

X_sample = np.random.normal(loc=theta_true, scale=np.sqrt(psi_true), size=n)

print(f"True Theta (Mean): {theta_true:.4f}")
print(f"True Psi (Variance): {psi_true:.4f}")
print(f"Generated Observations (X_i): \n{X_sample}")

In [ ]:
np.random.seed(42)

x_bar = np.mean(X_sample)
s2 = np.var(X_sample, ddof=1) if n > 1 else 0.0

def log_posterior(theta, psi, x_bar, s2, n, rho=0.5):

    if theta <= 0 or psi <= 0:
        return -np.inf
        
    log_theta = np.log(theta)
    log_psi = np.log(psi)
    
    prior_term = -log_theta - log_psi - (log_theta**2 - 2*rho*log_theta*log_psi + log_psi**2) / (2 * (1 - rho**2))
    
    likelihood_term = -(n/2)*np.log(psi) - (n*(x_bar - theta)**2 + (n-1)*s2) / (2*psi)
    
    return prior_term + likelihood_term

def mh_posterior_sampler(num_samples, x_bar, s2, n, init_theta, init_psi, step_size=0.5):
    samples = np.zeros((num_samples, 2))
    current_theta, current_psi = init_theta, init_psi
    current_log_prob = log_posterior(current_theta, current_psi, x_bar, s2, n)
    
    accepted = 0
    for step in range(num_samples):
        prop_theta = current_theta + np.random.normal(0, step_size)
        prop_psi = current_psi + np.random.normal(0, step_size)
        
        prop_log_prob = log_posterior(prop_theta, prop_psi, x_bar, s2, n)
        
        log_acc_ratio = prop_log_prob - current_log_prob
        

        if np.log(np.random.rand()) < log_acc_ratio:
            current_theta, current_psi = prop_theta, prop_psi
            current_log_prob = prop_log_prob
            accepted += 1
            
        samples[step] = [current_theta, current_psi]
        
    print(f"Acceptance Rate: {accepted / num_samples:.2f}")
    return samples

num_mcmc_samples = 10000

posterior_samples = mh_posterior_sampler(
    num_samples=num_mcmc_samples, 
    x_bar=x_bar, 
    s2=s2, 
    n=n, 
    init_theta=max(0.1, x_bar), 
    init_psi=max(0.1, s2),
    step_size=0.3 # You can tune this to get a better acceptance rate
)

burn_in = int(0.1 * num_mcmc_samples)
valid_samples = posterior_samples[burn_in:]

estimated_theta = np.mean(valid_samples[:, 0])
estimated_psi = np.mean(valid_samples[:, 1])

print(f"\n--- Results for n={n} ---")
print(f"True Theta : {theta_true:.4f} | MCMC Estimated Theta : {estimated_theta:.4f}")
print(f"True Psi   : {psi_true:.4f} | MCMC Estimated Psi   : {estimated_psi:.4f}")


n_values = [10, 100, 1000]
for n in n_values:
    print(f"\n=== Running MCMC for n={n} ===")
    
    X_sample = np.random.normal(loc=theta_true, scale=np.sqrt(psi_true), size=n)
    x_bar = np.mean(X_sample)
    s2 = np.var(X_sample, ddof=1) if n > 1 else 0.0
    
    posterior_samples = mh_posterior_sampler(
        num_samples=num_mcmc_samples, 
        x_bar=x_bar, 
        s2=s2, 
        n=n, 
        init_theta=max(0.1, x_bar), 
        init_psi=max(0.1, s2),
        step_size=0.3
    )
    
    burn_in = int(0.1 * num_mcmc_samples)
    valid_samples = posterior_samples[burn_in:]
    
    estimated_theta = np.mean(valid_samples[:, 0])
    estimated_psi = np.mean(valid_samples[:, 1])

    




## Exercise 7

In [ ]:
def tour_length(route, points):
    pts = points[route]
    diffs = pts - np.roll(pts, -1, axis=0)
    return np.sqrt((diffs ** 2).sum(axis=1)).sum()

def tour_cost(route, cost_matrix):
    return cost_matrix[route, np.roll(route, -1)].sum()

def propose_swap(route):
    new_route = route.copy()
    i, j = np.random.choice(len(route), size=2, replace=False)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

def propose_reverse(route):
    n = len(route)
    i, j = sorted(np.random.choice(n, size=2, replace=False))
    new_route = route.copy()
    new_route[i:j + 1] = new_route[i:j + 1][::-1]
    return new_route

def cooling_inv_sqrt(k):
    return 1.0 / np.sqrt(1 + k)

def cooling_inv_log(k):
    return 1.0 / np.log(2 + k)

def simulated_annealing_tsp(n_cities, cost_fn, n_iter, cooling, propose=propose_swap, seed=42):
    np.random.seed(seed)
    route = np.random.permutation(n_cities)
    cost = cost_fn(route)

    best_route, best_cost = route.copy(), cost
    costs = np.empty(n_iter)

    for k in range(n_iter):
        T = cooling(k)
        new_route = propose(route)
        new_cost = cost_fn(new_route)

        if new_cost < cost or np.random.uniform() < np.exp(-(new_cost - cost) / T):
            route, cost = new_route, new_cost
            if cost < best_cost:
                best_route, best_cost = route.copy(), cost

        costs[k] = cost

    return best_route, best_cost, costs

def plot_route(points, route, ax=None, title=None):
    """Plot the points and the closed tour connecting them in the given order."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    else:
        fig = ax.figure

    tour = points[np.append(route, route[0])]
    ax.plot(tour[:, 0], tour[:, 1], 'o-')
    ax.set_aspect('equal')
    if title:
        ax.set_title(title)
    return fig, ax

In [ ]:
# Random points on a circle
n = 40
theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
points = np.column_stack((np.cos(theta), np.sin(theta)))

n_iter = 5000
best_route, best_cost, costs = simulated_annealing_tsp(n, lambda r: tour_length(r, points), n_iter, cooling_inv_sqrt)

print(f"n = {n}, n_iter = {n_iter}")
print(f"Best tour length: {best_cost:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_route(points, best_route, ax=axes[0], title=f"Best route (length={best_cost:.3f})")
axes[1].plot(costs)
axes[1].set_xlabel("Iteration k")
axes[1].set_ylabel("Tour length")
axes[1].set_title(r"Cost during annealing, $T_k = 1/\sqrt{1+k}$")
fig.tight_layout()
plt.show()

In [ ]:
n_circle = 40
theta = np.linspace(0, 2 * np.pi, n_circle, endpoint=False)
circle_points = np.column_stack((np.cos(theta), np.sin(theta)))

best_route_c, best_cost_c, costs_c = simulated_annealing_tsp(n_circle, lambda r: tour_length(r, circle_points), n_iter, cooling_inv_log)

optimal_cost = tour_length(np.arange(n_circle), circle_points)
print(f"SA best tour length:      {best_cost_c:.4f}")
print(f"Optimal (ordered) length: {optimal_cost:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_route(circle_points, best_route_c, ax=axes[0], title=f"SA route on circle (length={best_cost_c:.3f})")
axes[1].plot(costs_c)
axes[1].set_xlabel("Iteration k")
axes[1].set_ylabel("Tour length")
axes[1].set_title(r"Cost during annealing, $T_k = 1/\ln(2+k)$")
fig.tight_layout()
plt.show()

In [ ]:
cost_matrix = np.loadtxt("cost.csv", delimiter=",")
n_cities = cost_matrix.shape[0]

print(f"Cost matrix: {n_cities} x {n_cities}, symmetric: {np.allclose(cost_matrix, cost_matrix.T)}")

n_iter = 5000
best_route, best_cost, costs = simulated_annealing_tsp(n_cities, lambda r: tour_cost(r, cost_matrix), n_iter, cooling_inv_sqrt, propose=propose_swap)

print(f"Best route: {best_route}")
print(f"Best cost:  {best_cost:.2f}")

plt.figure(figsize=(7, 4))
plt.plot(costs)
plt.xlabel("Iteration k")
plt.ylabel("Tour cost")
plt.title(r"Cost during annealing, swap proposal, $T_k = 1/\sqrt{1+k}$")
plt.tight_layout()
plt.show()

In [ ]:
configs = [
    ("swap, T=1/sqrt(1+k)", propose_swap, cooling_inv_sqrt),
    ("swap, T=1/ln(2+k)", propose_swap, cooling_inv_log),
    ("reverse, T=1/sqrt(1+k)", propose_reverse, cooling_inv_sqrt),
    ("reverse, T=1/ln(2+k)", propose_reverse, cooling_inv_log),
]

n_iter = 5000
results = {}
for name, propose, cooling in configs:
    _, best_cost_cfg, costs_cfg = simulated_annealing_tsp(
        n_cities, lambda r: tour_cost(r, cost_matrix), n_iter, cooling, propose=propose
    )
    results[name] = (best_cost_cfg, costs_cfg)
    print(f"{name:35s} best cost = {best_cost_cfg:.2f}")

plt.figure(figsize=(8, 5))
for name, (best_cost_cfg, costs_cfg) in results.items():
    plt.plot(costs_cfg, label=f"{name} (best={best_cost_cfg:.1f})")
plt.xlabel("Iteration k")
plt.ylabel("Tour cost")
plt.title("Cost during annealing for different proposals / cooling schedules")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Exercise 8

In [ ]:
xi = [56, 101, 78, 67, 93, 87, 64, 72, 80, 69]
n = len(xi)

a = -5 
b = 5

np.random.seed(42)
bootstrap_samples = 10000

original_mean = np.mean(xi)

bootstrap_differences = np.zeros(bootstrap_samples)

for i in range(bootstrap_samples):
    sample = np.random.choice(xi, size=n, replace=True)
    
    bootstrap_mean = np.mean(sample)
    
    bootstrap_differences[i] = bootstrap_mean - original_mean

successes = (bootstrap_differences > a) & (bootstrap_differences < b)

estimator_p = np.mean(successes)

print(f"Bootstrap Estimate of probability p: {estimator_p:.4f}")

plt.hist(bootstrap_differences, bins=20, color='skyblue', edgecolor='black')

plt.axvline(a, color='red', linestyle='dashed', linewidth=2, label=f'a = {a}')
plt.axvline(b, color='red', linestyle='dashed', linewidth=2, label=f'b = {b}')

plt.title("Bootstrap Distribution of Error (X̄* - X̄)")
plt.xlabel("Error Value")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.savefig("bootstrap_error_distribution.png")
plt.show()

ex 15

In [ ]:
import numpy as np

# Original data
data = np.array([5, 4, 9, 6, 21, 17, 11, 20, 7, 10, 21, 15, 13, 16, 8])
n = len(data)
B = 10000

np.random.seed(42)

bootstrap_samples = np.random.choice(data, size=(B, n), replace=True)

bootstrap_variances = np.var(bootstrap_samples, axis=1, ddof=1)

var_s2_boot = np.var(bootstrap_variances, ddof=1)

print(f"Simulated Bootstrap Estimate of Var(S^2): {var_s2_boot:.4f}")

plt.hist(bootstrap_variances, bins=20, color='lightcoral', edgecolor='black')
plt.title("Bootstrap Distribution of Sample Variance (S^2)")
plt.xlabel("Sample Variance Value")
plt.ylabel("Frequency")
plt.grid(True)
plt.savefig("bootstrap_variance_distribution.png")

part 3

In [ ]:
import numpy as np
from scipy.stats import pareto
def analyze_pareto_bootstrap(n=200, shape=1.05, scale=1.0, replicates=100):

    dist = pareto(b=shape, scale=scale)

    data = dist.rvs(size=n)
    

    sample_mean = np.mean(data)
    sample_median = np.median(data)

    boot_means = np.zeros(replicates)
    boot_medians = np.zeros(replicates)

    for i in range(replicates):
        # Sample with replacement
        boot_sample = np.random.choice(data, size=n, replace=True)
        boot_means[i] = np.mean(boot_sample)
        boot_medians[i] = np.median(boot_sample)

    var_mean = np.var(boot_means, ddof=1)
    var_median = np.var(boot_medians, ddof=1)

    conf_interval_mean = np.percentile(boot_means, [2.5, 97.5])
    conf_interval_median = np.percentile(boot_medians, [2.5, 97.5])

    print(f"--- Original Sample Statistics (n={n}) ---")
    print(f"Sample Mean:   {sample_mean:.4f}")
    print(f"Sample Median: {sample_median:.4f}\n")

    print(f"--- Confidence Intervals (95%) ---")
    print(f"CI for the Mean:   [{conf_interval_mean[0]:.4f}, {conf_interval_mean[1]:.4f}]")
    print(f"CI for the Median: [{conf_interval_median[0]:.4f}, {conf_interval_median[1]:.4f}]\n")

    print(f"--- Bootstrap Variance Estimates (k={replicates}) ---")
    print(f"Variance of the Mean:   {var_mean:.4f}")
    print(f"Variance of the Median: {var_median:.4f}")


np.random.seed(42)
analyze_pareto_bootstrap()